In [ ]:
import random

# Constants and Initialization
INITIAL_STATES = [0, 1, 2, 3, 4, 5]
GOAL = 5
# R = [
#     [-1, 0, -1, -1, -1, -1, -1, -1],
#     [0, -1, 0, -1, -1, -1, -1, -1],
#     [-1, 0, -1, 0, -1, -1, 0, -1],
#     [-1, -1, 0, -1, 0, -1, -1, -1],
#     [-1, -1, -1, 0, -1, 100, -1, -1],
#     [-1, -1, -1, -1, 0, 100, -1, -1],
#     [-1, -1, 0, -1, -1, -1, -1, 0],
#     [-1, -1, -1, -1, -1, -1, 0, -1]
# ]

R = [
    [-1, -1, -1, -1,   0,  -1],  # Room 0 -> 4
    [-1, -1, -1,  0,  -1, 0],  # Room 1 -> 3, 5 (Goal: 100)
    [-1, -1, -1,  0,  -1,  -1],  # Room 2 -> 3
    [-1,  0,  0, -1,   0,  -1],  # Room 3 -> 1, 2, 4 (Two-way fixed)
    [ 0, -1, -1,  0,  -1, 0],  # Room 4 -> 0, 3, 5 (Goal: 100)
    [-1,  0, -1, -1,   0, 0]   # Room 5 -> 1, 4, 5 (Self-loop: 100)
]

GAMMA = 0.8
ITERATIONS = 10
Q_SIZE = len(INITIAL_STATES)
q = [[0 for _ in range(Q_SIZE)] for _ in range(Q_SIZE)]
currentState = 0

def setGoal(goal):
    if goal not in INITIAL_STATES:
        return False

    global R
    R = [row[:] for row in BASE_MAP]

    for row in range(len(R)):
        if R[row][goal] == 0:
            R[row][goal] = 100

    R[goal][goal] = 100

    return True

setGoal(GOAL)

def train():
    initialize()

    # Perform training, starting at all initial states.
    for _ in range(ITERATIONS):
        for state in INITIAL_STATES:
            episode(state)
            print("Q Matrix values:")
            for row in q:
                print("\t".join(map(str, row)))
            print()
                

    print("Final Q Matrix values:")
    for row in q:
        print("\t".join(map(str, row)))
    print()


def test():
    # Perform tests, starting at all initial states.
    print("Shortest routes from initial states:")
    for state in INITIAL_STATES:
        global currentState
        currentState = state
        newState = 0
        while currentState != GOAL:
            newState = maximum(currentState, True)
            print(f"{currentState}, ", end="")
            currentState = newState
        print(GOAL)


def episode(initialState):
    global currentState
    currentState = initialState

    # Travel from state to state until goal state is reached.
    while currentState != GOAL:
        choose_an_action()

    # When currentState = GOAL, Run through the set once more for convergence.
    for _ in range(Q_SIZE):
        choose_an_action()


def choose_an_action():
    global currentState
    possibleAction = get_random_action(Q_SIZE)

    if R[currentState][possibleAction] >= 0:
        q[currentState][possibleAction] = reward(possibleAction)
        currentState = possibleAction


def get_random_action(upperBound):
    action = 0
    choiceIsValid = False

    while not choiceIsValid:
        action = random.randint(0, upperBound - 1)
        if R[currentState][action] > -1:
            choiceIsValid = True

    return action


def initialize():
    global q
    q = [[0 for _ in range(Q_SIZE)] for _ in range(Q_SIZE)]


def maximum(state, returnIndexOnly):
    winner = 0
    done = False

    while not done:
        foundNewWinner = False
        for i in range(Q_SIZE):
            if i != winner:  # Avoid self-comparison.
                if q[state][i] > q[state][winner]:
                    winner = i
                    foundNewWinner = True

        if not foundNewWinner:
            done = True

    if returnIndexOnly:
        return winner
    else:
        return q[state][winner]


def reward(action):
    return int(R[currentState][action] + (GAMMA * maximum(action, False)))


if __name__ == "__main__":
    train()
    test()
